## Importing necessary libraries

In [1]:
import pandas as pd
import numpy as np
from scipy.stats import skew, kurtosis
from scipy.integrate import trapezoid
import re
import os
import matplotlib.pyplot as plt

#### A dictionary with key as file name and values is sub-dictionary with key as sensor names and values as list of indexes which are bad data and to be removed

In [2]:
# 0 indexed
bad = {"1_12-08-2025_OCP_Calibration_CaCl2.csv": {'R91':[0,1,2,3], 'R92':[0,1,2,3], 'R93':[0,1,2,3], 'R94':[0,1,2], 'R95':[0,1,2,3], 'R96':[0,1,2,3], 'R97':[0,1,2,3], 'R98':[0,1,2,3], 'R83':[0,1,2,3],
                                                  'R84':[0,1,2], 'R85':[0,1,2,3,4,5,6], 'R86':[0,1,2,3,4,5,6], 'R87':[0,1,2,3,4], 'R88':[0,1,2,3,4], 'R89':[0,1,2,3,4]},
       
       "2_12-16-2025_OCP_Calibration_CaCl2.csv": {'R91':[0,1,2,3,4,5,6], 'R92':[0,1,2,3], 'R93':[0,1,2,3,6], 'R94':[0,1,2], 'R95':[0,1,2,3,6], 'R96':[0,1,2], 'R97':[0,1,2], 'R98':[0,1,2], 'R83':[3],
                                                  'R84':[0], 'R85':[1,2,3], 'R86':[1,2,3], 'R87':[0,1,2,3], 'R88':[0], 'R89':[0]},
       
       "3_02-10-2026_OCP_Calibration_CaCl2.csv": {'R77':[0,1,2,3,4,5,6], 'R78':[0,1,2,3,4,5,6], 'R80':[6], 'R83':[4,5,6]},
       
       "4_04-21-2026_OCP_Calibration_CaCl2.csv": {'R78':[0,1,2,3,4,5,6], 'R80':[0,1,2], 'R83':[0], 'R92':[0], 'R98':[0], 'R102':[0,5,6], 'R103':[0,4,5,6], 'R104':[0,1,2,3,4,5,6], 'R106':[3,4,5,6],
                                                  'R116':[0,3,4,5,6], 'R117':[0,1,2,3,4,5,6], 'R118':[0,3,4,5,6], 'R121':[0], 'R123':[0,5,6], 'R99':[0,1,2,3,4,5,6], 'R101':[0,1,2,3,4,5,6]},
    
       "5_06-17-2026_OCP_Calibration_CaCl2.csv": {'R77':[6], 'R78':[6], 'R80':[0,6], 'R83':[6], 'R92':[5], 'R102':[0], 'R105':[0], 'R120':[0], 'R122':[0], 'R124':[0,5,6], 'R125':[0,5,6],
                                                  'R126':[0,1,2,3,4,5,6], 'R129':[0], 'R127':[0,1], 'R128':[0]},
    }

#### Function to make (start_time, end_time) tuples for each file

In [3]:
def build_windows(boundaries):
    return list(zip(boundaries[:-1], boundaries[1:]))
time_list = [[0,0.511111,1.025,1.5375,2.05,2.58194,3.1,np.inf], [0,0.506944,1.04722,1.57083,2.0875,2.62361,3.14444,np.inf], [0,0.53194,0.9,1.42222,1.96528,2.48472,3.00833,np.inf],
             [0,0.569444,1.17778,1.67778,2.21944,2.77778,3.39444,np.inf], [0,0.505556,1.00556,1.50278,2.00278,2.50556,3.00833,np.inf]]
file_list = ["1_12-08-2025_OCP_Calibration_CaCl2.csv", "2_12-16-2025_OCP_Calibration_CaCl2.csv", "3_02-10-2026_OCP_Calibration_CaCl2.csv", "4_04-21-2026_OCP_Calibration_CaCl2.csv",
             "5_06-17-2026_OCP_Calibration_CaCl2.csv"]
time_step = {}
for i, j in zip(file_list, time_list):
    time_step[i] = build_windows(j)

In [4]:
print(time_step)
print(bad)

{'1_12-08-2025_OCP_Calibration_CaCl2.csv': [(0, 0.511111), (0.511111, 1.025), (1.025, 1.5375), (1.5375, 2.05), (2.05, 2.58194), (2.58194, 3.1), (3.1, inf)], '2_12-16-2025_OCP_Calibration_CaCl2.csv': [(0, 0.506944), (0.506944, 1.04722), (1.04722, 1.57083), (1.57083, 2.0875), (2.0875, 2.62361), (2.62361, 3.14444), (3.14444, inf)], '3_02-10-2026_OCP_Calibration_CaCl2.csv': [(0, 0.53194), (0.53194, 0.9), (0.9, 1.42222), (1.42222, 1.96528), (1.96528, 2.48472), (2.48472, 3.00833), (3.00833, inf)], '4_04-21-2026_OCP_Calibration_CaCl2.csv': [(0, 0.569444), (0.569444, 1.17778), (1.17778, 1.67778), (1.67778, 2.21944), (2.21944, 2.77778), (2.77778, 3.39444), (3.39444, inf)], '5_06-17-2026_OCP_Calibration_CaCl2.csv': [(0, 0.505556), (0.505556, 1.00556), (1.00556, 1.50278), (1.50278, 2.00278), (2.00278, 2.50556), (2.50556, 3.00833), (3.00833, inf)]}
{'1_12-08-2025_OCP_Calibration_CaCl2.csv': {'R91': [0, 1, 2, 3], 'R92': [0, 1, 2, 3], 'R93': [0, 1, 2, 3], 'R94': [0, 1, 2], 'R95': [0, 1, 2, 3], 'R96'

#### Function to replace file wise bad data with NaN values for each sensor

In [5]:
def remove_bad_regions(path, file_name, bad, time_step, time_col="t_s", save_dir="cleaned_data"):
    """
    Parameters
    df : pd.DataFrame - Measurement dataframe.
    file_name : str - Name of the current file.
    bad : dict - Dictionary of bad windows.
    time_step : dict - Dictionary mapping file -> [(start,end), ...]
    time_col : str - Time column.
    """
    df = pd.read_csv(path) # raw cleaned df
    cols = ["t_s"] # keep time column
    sensor_cols = [c for c in df.columns if re.fullmatch(r"R\d+", c)] # keep only sensor columns (R75, R76, ...)
    cols.extend(sensor_cols)
    df = df[cols].copy()
    
    if file_name not in time_step:
        raise KeyError(f"No switch times defined for {file_name}. Add it to time_list.")
    
    windows = time_step[file_name] # list of tuples of time steps
    for sensor, bad_idx in bad.get(file_name, {}).items(): # sensor => sensor name, bad_idx => list of indexes to be removed
        if sensor not in df.columns:
            continue
        for idx in bad_idx:
            if idx >= len(windows):
                print(f"Skipping bad index {idx} for {sensor} => only {len(windows)} windows")
                continue
            start, end = windows[idx] # start & end time to remove as per indexes in bad
            mask = ((df[time_col]/3600) >= start) & ((df[time_col]/3600) < end) # array having True values where cond is True for every row
            df.loc[mask, sensor] = np.nan # modify value to nan where mask is True
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, file_name)
    df.to_csv(save_path, index=False)
    print(f"Saved: {save_path}")
    return df

#### Removing bad data from all files present and saved those cleaned file into new folder name "cleaned_data"

In [6]:
from pathlib import Path
for file in file_list:
    remove_bad_regions(file, file, bad=bad, time_step=time_step, time_col='t_s', save_dir="cleaned_data")

saved = {p.name for p in Path("cleaned_data").glob("*.csv")}
missing = set(file_list) - saved
print(f"{len(saved)}/{len(file_list)} saved", f"| MISSING: {missing}" if missing else "| all good")

Saved: cleaned_data/1_12-08-2025_OCP_Calibration_CaCl2.csv
Saved: cleaned_data/2_12-16-2025_OCP_Calibration_CaCl2.csv
Saved: cleaned_data/3_02-10-2026_OCP_Calibration_CaCl2.csv
Saved: cleaned_data/4_04-21-2026_OCP_Calibration_CaCl2.csv
Saved: cleaned_data/5_06-17-2026_OCP_Calibration_CaCl2.csv
5/5 saved | all good


#### This is just a try run to see if it has removed those bad data or not

In [11]:
df1 = pd.read_csv(r'../clean_name_dataset/cleaned_data/1_12-08-2025_OCP_Calibration_CaCl2.csv')

In [12]:
df1.shape

(2618, 25)

In [13]:
df1.head()

,t_s,R91,R92,R93,R94,R95,R96,R97,R98,R83,...,R89,R90,R75,R76,R77,R78,R79,R80,R81,R82
0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.013607,0.005171,-0.002260,0.025918,0.027570,0.027524,0.001611,0.026210,0.024840
1,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.013838,0.005807,-0.001966,0.026329,0.027965,0.027779,0.001975,0.026547,0.024927
2,10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.013993,0.006368,-0.001742,0.026658,0.028295,0.028004,0.002289,0.026796,0.024998
3,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.014066,0.006872,-0.001565,0.026912,0.028557,0.028194,0.002564,0.027014,0.025046
4,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,0.014130,0.007313,-0.001422,0.027115,0.028774,0.028367,0.002807,0.027222,0.025079


#### Now after removing bad data this fucntion reads a clean file and converts potential in mV

In [14]:
def read_sensor_file(path):
    '''
    Parameters
    path : path of folder in which file is present
    '''
    df = pd.read_csv(path)
    # keep time column
    cols = ["t_s"]

    # keep only sensor columns (R75, R76, ...)
    sensor_cols = [c for c in df.columns if re.fullmatch(r"R\d+", c)]
    cols.extend(sensor_cols)
    df = df[cols].copy()

    # convert V -> mV & inverting sign
    df[sensor_cols] *(-1000)
    return df

#### Function to extract features for each concentration window

In [15]:
def extract_features(df, windows):
    '''
    Parameters
    df : pd.DataFrame - Measurement dataframe.
    windows : time_step which e have built earlier using build_window function
    '''
    
    sensors = [c for c in df.columns if c.startswith("R")]
    rows = []

    for sensor in sensors:
        previous_mean = np.nan
        for step, w in enumerate(windows):
            #print(w[0], w[1])
            seg = df.loc[(df["t_s"]/3600 >= w[0]) & (df["t_s"]/3600 < w[1]), ["t_s", sensor]].dropna()
            if len(seg) < 5: # length of segment should be atleast 5 i.e it should contains atleast 5 rows
                continue    

            t_hr = seg["t_s"].to_numpy() / 3600
            y = seg[sensor].to_numpy()   # no *(-1000) if already converted in read_sensor_file()
            dy = np.gradient(y, t_hr)
            
            features = {"sensor": sensor, "step": step+1, 'time_start': f'{t_hr[0]}', 'time_end': f'{t_hr[-1]}',
                # plateau features
                "mean": np.mean(y), # average of all points
                "median": np.median(y), # median of all points
                "last": y[-1], # last point
                "min": np.min(y), # min value
                "max": np.max(y), # max value
                "range": np.ptp(y), # range of value
                "std": np.std(y), # standard deviation
                "var": np.var(y), # variance

                # dynamics
                "max_slope": np.max(dy), # max slope
                "min_slope": np.min(dy), # min slope
                "mean_slope": np.mean(dy), # mean of all slopes

                # drift
                "drift": y[-1]-y[0],

                # noise
                "mad": np.mean(np.abs(y-np.mean(y))), # mean absolute deviation
                "mead" : np.median(np.abs(y - np.median(y))), # median absolute deviation
                "rms": np.sqrt(np.mean(y**2)), # root mean square value

                # area
                "auc": trapezoid(y,t_hr), # area under the curve

                # statistics
                #"skew": skew(y), # measures asymmetry of signal
                #"kurtosis": kurtosis(y), # measures how heavy tails are, or how prone signal is to extreme values

                # transition
            #     "delta_previous":
            #         np.nan if np.isnan(previous_mean)
            #         else np.mean(y)-previous_mean
            }

            previous_mean = np.mean(y)
            rows.append(features)

    return pd.DataFrame(rows)

#### This is where we actually extract out features and store them into a file name features_dataset.cav

In [18]:
folder = r"../clean_name_dataset/cleaned_data"

all_features = []
for file in Path(folder).glob("*.csv"):
    print(file.name)
    df = read_sensor_file(file)
    windows = time_step[file.name]
    feat = extract_features(df, windows)
    feat["file"] = file.name
    all_features.append(feat)

dataset = pd.concat(all_features, ignore_index=True)
dataset.to_csv("features_dataset.csv", index=False)

3_02-10-2026_OCP_Calibration_CaCl2.csv
1_12-08-2025_OCP_Calibration_CaCl2.csv
5_06-17-2026_OCP_Calibration_CaCl2.csv
2_12-16-2025_OCP_Calibration_CaCl2.csv
4_04-21-2026_OCP_Calibration_CaCl2.csv


In [19]:
dataset.columns # command to see name of all columnd present in dataset

Index(['sensor', 'step', 'time_start', 'time_end', 'mean', 'median', 'last',
       'min', 'max', 'range', 'std', 'var', 'max_slope', 'min_slope',
       'mean_slope', 'drift', 'mad', 'mead', 'rms', 'auc', 'file'],
      dtype='str')

In [20]:
dataset['sensor'].unique() # command to see all unique values present in 'sensor' column of dataset dataframe

<StringArray>
[ 'R75',  'R80',  'R81',  'R83',  'R92',  'R98', 'R102', 'R103', 'R104',
 'R105', 'R106', 'R116', 'R117', 'R118', 'R119', 'R120', 'R121', 'R122',
 'R123',  'R91',  'R93',  'R94',  'R95',  'R96',  'R97',  'R84',  'R87',
  'R88',  'R89',  'R90',  'R76',  'R77',  'R78',  'R79',  'R82', 'R100',
 'R114', 'R124', 'R125', 'R127', 'R128', 'R129',  'R85',  'R86']
Length: 44, dtype: str

In [21]:
dataset

,sensor,step,time_start,time_end,mean,median,last,min,max,range,...,var,max_slope,min_slope,mean_slope,drift,mad,mead,rms,auc,file
0,R75,1,0.0,0.5305555555555556,0.004492,0.004492,0.004492,0.004364,0.004570,0.000206,...,9.217131e-11,0.046122,-2.815247e-02,0.000285,0.000128,0.000001,0.000000,0.004492,0.002383,3_02-10-2026_OCP_Calibration_CaCl2.csv
1,R75,2,0.5319444444444444,0.8986111111111111,0.015575,0.016680,0.016680,-0.001289,0.016758,0.018047,...,6.089916e-06,2.193832,-4.162445e+00,0.025261,0.012188,0.001529,0.000078,0.015770,0.005718,3_02-10-2026_OCP_Calibration_CaCl2.csv
2,R75,3,0.9,1.4208333333333334,-0.002936,-0.002070,-0.001289,-0.020273,-0.001289,0.018984,...,6.413908e-06,1.968613,-1.110223e-16,0.038970,0.018984,0.001543,0.000546,0.003877,-0.001518,3_02-10-2026_OCP_Calibration_CaCl2.csv
3,R75,4,1.4222222222222223,1.9652777777777777,-0.048164,-0.048243,-0.094648,-0.094648,-0.043477,0.051171,...,6.023583e-06,2.699890,-3.459389e+01,-0.128572,-0.045781,0.000600,0.000235,0.048227,-0.026123,3_02-10-2026_OCP_Calibration_CaCl2.csv
4,R75,5,1.9666666666666666,2.4833333333333334,-0.090532,-0.090820,-0.091133,-0.092930,-0.088789,0.004141,...,4.114225e-07,0.843887,-2.815247e-02,0.004599,0.001797,0.000504,0.000078,0.090534,-0.046773,3_02-10-2026_OCP_Calibration_CaCl2.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
568,R122,7,3.3944444444444444,3.9166666666666665,-0.172392,-0.172462,-0.170978,-0.173243,-0.170978,0.002265,...,4.551764e-07,0.055962,-4.205704e-02,0.001710,0.000859,0.000589,0.000625,0.172393,-0.090030,4_04-21-2026_OCP_Calibration_CaCl2.csv
569,R123,2,0.5694444444444444,1.1777777777777778,0.021181,0.021367,0.013243,0.013243,0.023086,0.009843,...,2.641233e-06,0.028152,-1.968613e+00,-0.020134,-0.009609,0.001421,0.001562,0.021244,0.012894,4_04-21-2026_OCP_Calibration_CaCl2.csv
570,R123,3,1.1805555555555556,1.6777777777777778,0.014964,0.015196,-0.014805,-0.014805,0.015586,0.030391,...,5.117293e-06,0.042229,-1.071579e+01,-0.086094,-0.028203,0.000493,0.000235,0.015134,0.007484,4_04-21-2026_OCP_Calibration_CaCl2.csv
571,R123,4,1.6805555555555556,2.216666666666667,-0.010807,-0.010547,-0.008008,-0.014727,-0.008008,0.006719,...,3.793013e-06,0.112438,0.000000e+00,0.012468,0.006719,0.001692,0.001679,0.010981,-0.005792,4_04-21-2026_OCP_Calibration_CaCl2.csv


#### map to describe log(concentration) values for each file

In [22]:
logC_map = {"1_12-08-2025_OCP_Calibration_CaCl2.csv": {1:-8, 2:-7, 3:-6, 4:-5, 5:-4, 6:-3, 7:-2},
            "2_12-16-2025_OCP_Calibration_CaCl2.csv": {1:-8, 2:-7, 3:-6, 4:-5, 5:-4, 6:-3, 7:-2},
            "3_02-10-2026_OCP_Calibration_CaCl2.csv": {1:-7, 2:-6, 3:-5, 4:-4, 5:-3, 6:-2, 7:-1},
            "4_04-21-2026_OCP_Calibration_CaCl2.csv": {1:-7, 2:-6, 3:-5, 4:-4, 5:-3, 6:-2, 7:-1},
            "5_06-17-2026_OCP_Calibration_CaCl2.csv": {1:-7, 2:-6, 3:-5, 4:-4, 5:-3, 6:-2, 7:-1}
        }

#### Add log(concentration) column to our dataset dataframe

In [23]:
dataset["logC"] = dataset.apply(lambda row: logC_map[row["file"]][row["step"]], axis=1)

In [24]:
dataset.head() # command to see 1st five rows of dataframe

,sensor,step,time_start,time_end,mean,median,last,min,max,range,...,max_slope,min_slope,mean_slope,drift,mad,mead,rms,auc,file,logC
0,R75,1,0.0,0.5305555555555556,0.004492,0.004492,0.004492,0.004364,0.004570,0.000206,...,0.046122,-2.815247e-02,0.000285,0.000128,0.000001,0.000000,0.004492,0.002383,3_02-10-2026_OCP_Calibration_CaCl2.csv,-7
1,R75,2,0.5319444444444444,0.8986111111111111,0.015575,0.016680,0.016680,-0.001289,0.016758,0.018047,...,2.193832,-4.162445e+00,0.025261,0.012188,0.001529,0.000078,0.015770,0.005718,3_02-10-2026_OCP_Calibration_CaCl2.csv,-6
2,R75,3,0.9,1.4208333333333334,-0.002936,-0.002070,-0.001289,-0.020273,-0.001289,0.018984,...,1.968613,-1.110223e-16,0.038970,0.018984,0.001543,0.000546,0.003877,-0.001518,3_02-10-2026_OCP_Calibration_CaCl2.csv,-5
3,R75,4,1.4222222222222223,1.9652777777777777,-0.048164,-0.048243,-0.094648,-0.094648,-0.043477,0.051171,...,2.699890,-3.459389e+01,-0.128572,-0.045781,0.000600,0.000235,0.048227,-0.026123,3_02-10-2026_OCP_Calibration_CaCl2.csv,-4
4,R75,5,1.9666666666666666,2.4833333333333334,-0.090532,-0.090820,-0.091133,-0.092930,-0.088789,0.004141,...,0.843887,-2.815247e-02,0.004599,0.001797,0.000504,0.000078,0.090534,-0.046773,3_02-10-2026_OCP_Calibration_CaCl2.csv,-3


In [25]:
dataset.shape # command to see (rows, columns) of dataframe

(573, 22)

#### Import ML models, evaluation metircs, scaling, etc..

In [26]:
from sklearn.ensemble import (ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor)
from sklearn.linear_model import RidgeCV
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

### Training model with level & shape features

In [27]:
level = ["mean", "median", "last", "min", "max", "rms"] # describes absolute potential and depends heavily on electrodes
shape = ["range", "std", "max_slope", "min_slope", "mean_slope", "drift", "mad", "mead"] # describe response shape

for c in level:
    dataset[c + "_ref"] = dataset[c] - dataset.groupby(["file", "sensor"])[c].transform("mean") # done so that offset of each sensor is removed

features = [c + "_ref" for c in level] + shape

In [28]:
X = dataset[features].values # columns which will be used as features
y = dataset['logC'].values # column used as target

In [29]:
print(X.shape, y.shape)

(573, 14) (573,)


#### Whole piece of code to train every model in 2 different ways : 1. holding out a sensor, 2. holding out a whole calibration run

In [30]:
# --- models -------------------------------------------------------------
MODELS = {
    "Ridge":      make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
    "kNN":        make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "SVR-rbf":    make_pipeline(StandardScaler(), SVR(C=10)),
    "RF":         RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "HistGB":     HistGradientBoostingRegressor(random_state=0),
}

# --- grouped CV: no sensor or run  in both train and test --------------
for group_col in ["sensor", "file"]:
    groups = dataset[group_col].values
    rows = []
    for name, model in MODELS.items():
        pred = cross_val_predict(model, X, y, cv=GroupKFold(n_splits=5), groups=groups)
        rows.append({"model": name, "R2": r2_score(y, pred), "MAE_dec": mean_absolute_error(y, pred),
                     "RMSE_dec": np.sqrt(mean_squared_error(y, pred))})
    print(f"\n=== held-out {group_col} ===")
    print(pd.DataFrame(rows).sort_values("RMSE_dec").to_string(index=False, float_format="%.3f"))


=== held-out sensor ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.901    0.401     0.605
    HistGB 0.892    0.439     0.630
        RF 0.888    0.435     0.642
   SVR-rbf 0.858    0.500     0.722
       kNN 0.855    0.504     0.731
     Ridge 0.753    0.674     0.954

=== held-out file ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.773    0.702     0.915
   SVR-rbf 0.762    0.718     0.936
    HistGB 0.759    0.726     0.941
        RF 0.756    0.723     0.948
       kNN 0.741    0.734     0.976
     Ridge 0.726    0.738     1.004


### Training model with level features only

In [31]:
features = [c + "_ref" for c in level]
X = dataset[features].values
y = dataset['logC'].values

# --- models -------------------------------------------------------------
MODELS = {
    "Ridge":      make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
    "kNN":        make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "SVR-rbf":    make_pipeline(StandardScaler(), SVR(C=10)),
    "RF":         RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "HistGB":     HistGradientBoostingRegressor(random_state=0),
}

# --- grouped CV: no sensor or run  in both train and test --------------
for group_col in ["sensor", "file"]:
    groups = dataset[group_col].values
    rows = []
    for name, model in MODELS.items():
        pred = cross_val_predict(model, X, y, cv=GroupKFold(5), groups=groups)
        rows.append({"model": name, "R2": r2_score(y, pred), "MAE_dec": mean_absolute_error(y, pred),
                     "RMSE_dec": np.sqrt(mean_squared_error(y, pred))})
    print(f"\n=== held-out {group_col} ===")
    print(pd.DataFrame(rows).sort_values("RMSE_dec").to_string(index=False, float_format="%.3f"))


=== held-out sensor ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.881    0.451     0.661
        RF 0.869    0.479     0.694
       kNN 0.865    0.483     0.705
    HistGB 0.858    0.519     0.723
   SVR-rbf 0.844    0.553     0.758
     Ridge 0.776    0.699     0.908

=== held-out file ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.799    0.654     0.861
        RF 0.793    0.656     0.874
    HistGB 0.787    0.670     0.886
   SVR-rbf 0.784    0.695     0.892
     Ridge 0.779    0.699     0.903
       kNN 0.772    0.679     0.916


### Training model with level & 2 extra features

In [32]:
# how much the voltage changed coming into this step, and how much it changes going out of it
dataset = dataset.sort_values(["file","sensor","step"]).reset_index(drop=True)
g = dataset.groupby(["file","sensor"])["mean"]
gap = dataset["step"] - dataset.groupby(["file","sensor"])["step"].shift(1)
dp = dataset["mean"] - g.shift(1) # jump up from previous step
dp = dp / gap
dn = g.shift(-1) - dataset["mean"] # jump to next step
dn = dn / gap
# shift(1) slides each block down by one row, so each row now sees the previous step's voltage

# fill the missing edges with that sensor's median jump, NOT with 0
med = dp.groupby([dataset['file'], dataset['sensor']]).transform("median")
dataset["d_prev"] = dp.fillna(med)
dataset["d_next"] = dn.fillna(med)

In [33]:
features = [c + "_ref" for c in level] + ["d_prev", "d_next"]
X = dataset[features].values
y = dataset['logC'].values

# --- models -------------------------------------------------------------
MODELS = {
    "Ridge":      make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-3, 3, 13))),
    "kNN":        make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=5)),
    "SVR-rbf":    make_pipeline(StandardScaler(), SVR(C=10)),
    "RF":         RandomForestRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "ExtraTrees": ExtraTreesRegressor(n_estimators=300, random_state=0, n_jobs=-1),
    "HistGB":     HistGradientBoostingRegressor(random_state=0),
}

# --- grouped CV: no sensor or run  in both train and test --------------
for group_col in ["sensor", "file"]:
    groups = dataset[group_col].values
    rows = []
    for name, model in MODELS.items():
        pred = cross_val_predict(model, X, y, cv=GroupKFold(5), groups=groups)
        rows.append({"model": name, "R2": r2_score(y, pred), "MAE_dec": mean_absolute_error(y, pred),
                     "RMSE_dec": np.sqrt(mean_squared_error(y, pred))})
    print(f"\n=== held-out {group_col} ===")
    print(pd.DataFrame(rows).sort_values("RMSE_dec").to_string(index=False, float_format="%.3f"))


=== held-out sensor ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.936    0.285     0.487
    HistGB 0.923    0.351     0.533
        RF 0.919    0.321     0.547
   SVR-rbf 0.915    0.378     0.560
       kNN 0.907    0.317     0.584
     Ridge 0.820    0.643     0.815

=== held-out file ===
     model    R2  MAE_dec  RMSE_dec
ExtraTrees 0.875    0.475     0.679
       kNN 0.871    0.439     0.690
    HistGB 0.865    0.495     0.706
   SVR-rbf 0.864    0.548     0.709
        RF 0.856    0.493     0.727
     Ridge 0.814    0.654     0.827


In [34]:
import sys, importlib.metadata as md
pkgs = ["pandas","numpy","scikit-learn","scipy","matplotlib","streamlit","joblib"]
print("python", sys.version)
for p in pkgs:
    try: print(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError: print(f"{p}  NOT INSTALLED")

python 3.12.7 (v3.12.7:0b05ead877f, Sep 30 2024, 23:18:00) [Clang 13.0.0 (clang-1300.0.29.30)]
pandas==3.0.5
numpy==2.5.1
scikit-learn==1.9.0
scipy==1.18.0
matplotlib==3.11.1
streamlit  NOT INSTALLED
joblib==1.5.3
